# 04 · MLP 베이스라인 + 불균형 대응

> **역할**: 딥러닝 '바닥(베이스라인)'을 세우고, 백오더 0.67%라는 극단 불균형을
> 어떤 방법으로 다뤄야 하는지 **한 번에 하나씩 바꿔가며 실험(Ablation)**한다.
>
> 핵심 규칙: **한 번에 딱 하나만 바꾸고, 똑같은 지표로 재서, 결과표(results.csv)에 한 줄씩 쌓는다.**
> 그 표가 곧 발표의 뼈대.

실험: ① 그냥 학습(붕괴 확인) → ② 가중치 → ③ Focal → ④ +Dropout → ⑤ +BatchNorm → ⑥ 언더샘플링

## 0. 환경 설정
- **torch (PyTorch)**: 신경망을 만들고 학습시키는 딥러닝 프레임워크. 텐서(다차원 배열) 연산·자동미분·최적화를 담당.
- `load_processed`로 02에서 저장한 데이터를 그대로 불러온다 (전처리 재실행 안 함).

In [8]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from utils import set_seed, compute_metrics, log_result, load_processed
set_seed(42)

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
OUT_DIR     = PROJECT_ROOT / "processed"
RESULTS_CSV = PROJECT_ROOT / "notebooks" / "results.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
device: cpu


## 1. 데이터 불러오기
02에서 만든 train/val/test를 불러와 X(피처)와 y(정답)로 나눈다.
신경망은 숫자만 먹으므로 float32 배열로 바꾼다.

In [9]:
train_df, val_df, test_df = load_processed(OUT_DIR)
TARGET = "went_on_backorder"
feature_cols = [c for c in train_df.columns if c != TARGET]

X_train = train_df[feature_cols].values.astype("float32")
y_train = train_df[TARGET].values.astype("float32")
X_val   = val_df[feature_cols].values.astype("float32")
y_val   = val_df[TARGET].values.astype("float32")

print("피처 수:", len(feature_cols))
print("train 양성:", int(y_train.sum()), "/", len(y_train))

피처 수: 33
train 양성: 9034 / 1350288


## 2. 텐서로 바꾸고 배치로 묶기
PyTorch는 '텐서'로 계산한다. 데이터를 텐서로 바꾸고, 한 번에 다 넣지 않고
2048개씩 잘라서(batch) 학습한다 (메모리 절약 + 학습 안정).
- `shuffle=True`(학습용)일 때만 마지막 자투리 배치를 버린다(`drop_last`) — BatchNorm이 1개짜리 배치에서 에러나는 걸 방지.

In [10]:
from torch.utils.data import TensorDataset, DataLoader

def make_loader(X, y, batch_size=2048, shuffle=False):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)

# val은 평가용이라 한 번에 올려둔다
X_val_t = torch.from_numpy(X_val).to(device)

## 3. MLP 모델 (단순하게)
입력(피처 33개) -> 64 -> 32 -> 출력 1개(백오더 점수).
Dropout / BatchNorm은 **켜고 끌 수 있게** 했다 (나중에 효과를 비교하려고).
- **Dropout**: 학습 중 뉴런 일부를 무작위로 꺼서 과적합을 막음 (한 책만 외우지 않게).
- **BatchNorm**: 층마다 값의 분포를 일정하게 맞춰 학습을 안정·가속.

In [11]:
def make_mlp(input_dim, use_dropout=False, use_batchnorm=False):
    layers = []
    sizes = [input_dim, 64, 32]          # 입력 -> 64 -> 32
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(sizes[i + 1]))
        layers.append(nn.ReLU())
        if use_dropout:
            layers.append(nn.Dropout(0.3))
    layers.append(nn.Linear(32, 1))      # 마지막: 점수 1개 (확률은 sigmoid로)
    return nn.Sequential(*layers)

## 4. Focal Loss
불균형이 심하면 모델이 그냥 전부 'No'로 찍어버린다. 이를 막는 두 장치:
- **가중 BCE**: 소수 클래스(Yes)를 틀리면 벌점을 크게 (아래 학습함수에서 `pos_weight` 한 줄).
- **Focal Loss**: 이미 잘 맞히는 쉬운 샘플은 거의 무시하고, 어려운 샘플에 집중.

In [12]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)        # 정답일 확률
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

## 5. 학습 함수 (모든 실험이 공통 사용)
모델과 '손실 방식'(bce / weighted / focal)만 바꿔 넣으면 학습하고, val 확률을 돌려준다.
함수로 묶어두면 실험마다 코드를 복붙할 필요가 없다.

In [13]:
def train_model(model, loader, loss_kind="bce", epochs=8, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # 손실 방식 고르기
    if loss_kind == "weighted":
        n_pos = y_train.sum()
        n_neg = len(y_train) - n_pos
        pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    elif loss_kind == "focal":
        loss_fn = FocalLoss()
    else:                                   # "bce" = 아무 대응 없음
        loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb).squeeze(1)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

    # val 확률 계산
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t).squeeze(1)
        val_prob = torch.sigmoid(val_logits).cpu().numpy()
    return val_prob

## 6. 실험 — 한 번에 하나씩 바꾸기
매번 `set_seed(42)`로 출발선을 똑같이 맞추고, 결과는 results.csv에 기록.
- **Exp1 (아무 대응 X)**: 모델이 전부 No로 찍어 **Recall이 0에 가깝게** 나온다 -> 불균형 처리가 왜 필요한지 증명.
- **Exp2~5**: 가중치 / Focal / +Dropout / +BatchNorm 을 하나씩 더한다.

> 참고: 여기 Recall/F1은 **threshold 0.5 기준(아직 안 맞춤)**이라 낮게 보일 수 있다.
> 공정한 비교는 **PR_AUC**로 본다. 운영점 threshold 최적화는 08번에서.

Focal Loss는 기존 Cross Entropy에 조절 인자(Modulating Factor)인 \((1 - p_t)^\gamma\) 를 추가한 형태입니다.

In [14]:
input_dim = len(feature_cols)
runs = {}   # 이름 -> (모델, val확률, 지표). 나중에 PR_AUC 최고를 고르려고 저장.

def run_experiment(name, model, loss_kind, loader=train_loader):
    val_prob = train_model(model, loader, loss_kind=loss_kind)
    m = compute_metrics(y_val, val_prob)
    log_result(name, m, path=str(RESULTS_CSV))
    runs[name] = (model, val_prob, m)
    print("{:24s}  PR_AUC {:.3f}   Recall {:.3f}   F1 {:.3f}".format(
          name, m["PR_AUC"], m["Recall"], m["F1"]))

set_seed(42); run_experiment("MLP_1_plain_BCE",     make_mlp(input_dim),              "bce")
set_seed(42); run_experiment("MLP_2_weighted",      make_mlp(input_dim),              "weighted")
set_seed(42); run_experiment("MLP_3_focal",         make_mlp(input_dim),              "focal")
set_seed(42); run_experiment("MLP_4_focal_dropout", make_mlp(input_dim, True, False), "focal")
set_seed(42); run_experiment("MLP_5_focal_drop_bn", make_mlp(input_dim, True, True),  "focal")

MLP_1_plain_BCE           PR_AUC 0.186   Recall 0.022   F1 0.041
MLP_2_weighted            PR_AUC 0.166   Recall 0.891   F1 0.086
MLP_3_focal               PR_AUC 0.193   Recall 0.198   F1 0.242
MLP_4_focal_dropout       PR_AUC 0.186   Recall 0.130   F1 0.188
MLP_5_focal_drop_bn       PR_AUC 0.164   Recall 0.004   F1 0.009


## 7. 데이터 쪽 방법 — 언더샘플링
지금까지는 '손실 함수'로 불균형에 대응했다. 이번엔 '데이터'로 대응:
다수(No)를 무작위로 줄여 소수(Yes)와 비율을 맞춘 뒤 학습.
**손실 방법 vs 데이터 방법**, 어느 쪽이 나은지 비교하려는 것.

In [15]:
# No를 줄여서 No:Yes = 3:1 로 맞춘다 (간단한 무작위 언더샘플링)
pos_idx = np.where(y_train == 1)[0]
neg_idx = np.where(y_train == 0)[0]

rng = np.random.default_rng(42)
neg_keep = rng.choice(neg_idx, size=len(pos_idx) * 3, replace=False)
keep = np.concatenate([pos_idx, neg_keep])
rng.shuffle(keep)

X_us = X_train[keep]
y_us = y_train[keep]
us_loader = make_loader(X_us, y_us, shuffle=True)
print("언더샘플링 후 행 수:", len(keep), " 양성비율: {:.1%}".format(y_us.mean()))

set_seed(42); run_experiment("MLP_6_undersample", make_mlp(input_dim), "bce", loader=us_loader)

언더샘플링 후 행 수: 36136  양성비율: 25.0%
MLP_6_undersample         PR_AUC 0.124   Recall 0.763   F1 0.116


(Sequential(
   (0): Linear(in_features=33, out_features=64, bias=True)
   (1): ReLU()
   (2): Linear(in_features=64, out_features=32, bias=True)
   (3): ReLU()
   (4): Linear(in_features=32, out_features=1, bias=True)
 ),
 array([0.4768538 , 0.02522162, 0.03843242, ..., 0.01594   , 0.354483  ,
        0.01114468], shape=(337572,), dtype=float32))

## 8. 결과 비교표
results.csv를 불러와 한눈에 비교. **PR_AUC가 높을수록 소수 클래스를 잘 잡는 것.**
(같은 모델을 여러 번 돌렸으면 최신 것만 남긴다.)

In [16]:
res = pd.read_csv(RESULTS_CSV)
res = res.drop_duplicates(subset="model", keep="last")
res[["model", "PR_AUC", "ROC_AUC", "Recall", "Precision", "F1"]]

,model,PR_AUC,ROC_AUC,Recall,Precision,F1
0,MLP_1_plain_BCE,0.1859,0.9411,0.0217,0.4298,0.0413
1,MLP_2_weighted,0.1662,0.9469,0.8907,0.0451,0.0859
2,MLP_3_focal,0.1927,0.9463,0.1983,0.3109,0.2422
3,MLP_4_focal_dropout,0.1859,0.9421,0.1301,0.3387,0.1880
4,MLP_5_focal_drop_bn,0.1638,0.9443,0.0044,0.2041,0.0087
5,MLP_6_undersample,0.1239,0.9328,0.7627,0.0625,0.1156


## 9. 베스트 모델 저장 (분석용 결과 파일)
실험 중 **PR_AUC가 가장 높은** MLP를 골라 저장한다 (이름이 아니라 숫자로 선택).
아래 두 파일은 '실행 파일'이 아니라 08_Decision_Analysis가 쓸 **분석용 저장 결과**다:
- mlp_best.pt : 최고 MLP의 가중치
- mlp_best_val_prob.npy : 그 모델이 val에 매긴 확률 (threshold·비용 분석용)

In [17]:
# PR_AUC가 가장 높은 실험을 숫자로 고른다 (이름의 best가 아니라)
best_name = None
best_pr = -1
for name in runs:
    pr = runs[name][2]["PR_AUC"]
    if pr > best_pr:
        best_pr = pr
        best_name = name

best_model, best_prob, best_metrics = runs[best_name]
print("최고 MLP:", best_name, " PR_AUC:", best_metrics["PR_AUC"])

torch.save(best_model.state_dict(), PROJECT_ROOT / "notebooks" / "mlp_best.pt")
np.save(PROJECT_ROOT / "notebooks" / "mlp_best_val_prob.npy", best_prob)
print("저장 완료: mlp_best.pt, mlp_best_val_prob.npy")

저장 완료: mlp_best.pt, mlp_best_val_prob.npy


---
### 정리
- MLP 베이스라인 + 불균형 대응을 **한 번에 하나씩** 비교 -> results.csv에 누적
- **Exp1이 전부 No로 붕괴**하는 걸 보여 불균형 처리의 필요성을 증명 (발표 포인트)
- **손실 방법(가중/Focal) vs 데이터 방법(언더샘플)** 비교 = Ablation 점수

다음: **05_FT_Transformer** / **06_TabNet** (같은 데이터·같은 평가로 비교).
주의: Recall/F1은 threshold 0.5 기준이라 낮게 보임 -> 진짜 운영점·비용 기준은 08번.